# R-Bot Operator v3 — Fine-tune (Colab GPU)

Entrena **Qwen2.5-3B-Instruct** con LoRA para el operador lab Occupancy:
- Intents ROS / teleop en JSON + **reply** natural
- **Secuencias** con conectores (primero, después, por último, finalmente…)
- Unidades: metros, cm, pulgadas, pies → `meters`
- Ángulos sin decir «grados» («gira 45 a la derecha»)
- **Sin** rutas a zonas nombradas (almacén, válvula…) → `unknown`

## Runtime
1. **Runtime → Change runtime type → T4 GPU** (o L4)
2. Ejecuta celdas en orden
3. Sube **`operator_v3_sft.jsonl`** (desde `ml/datasets/`) cuando `UPLOAD=True`

## Después en el PC
```bash
cd ~/Documents/Proyectos/rbot-industrial/ml/export
ollama create rbot-operator -f Modelfile.rbot-operator
# LLM_PROVIDER=ollama · OLLAMA_MODEL=rbot-operator
bash ~/Documents/Proyectos/start-rbot.sh
```

Guía: `ml/COLAB_NOW.md`


In [ ]:
# 0) GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Runtime → GPU (T4/L4)"
print("OK:", torch.cuda.get_device_name(0), "VRAM_GB",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# 1) Dependencias
%pip install -q -U "transformers>=4.44,<4.48" "peft>=0.12,<0.14" \
  "datasets>=2.20,<3" "trl>=0.9,<0.12" "accelerate>=0.33,<1.1" \
  sentencepiece einops


In [ ]:
# 2) Dataset SFT (Operator v3)
from pathlib import Path
import json
from datasets import Dataset

UPLOAD = True  # False solo si pegas JSONL abajo
DATA_PATH = Path("/content/operator_v3_sft.jsonl")

if UPLOAD:
    from google.colab import files
    print("Sube ml/datasets/operator_v3_sft.jsonl …")
    up = files.upload()
    name = next(iter(up))
    DATA_PATH.write_bytes(up[name])

rows = []
with DATA_PATH.open(encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print("ejemplos:", len(rows))
print("sample:", rows[0])

SYSTEM = (
    "Eres el copiloto del robot industrial R-Bot (Nexus/Create3 + ROS2). "
    "Clasifica la orden y responde en español. SOLO JSON: intent, action, parameters, confidence, reply. "
    "sequence = planes multi-paso o órdenes con distancia/ángulo (drive/turn/wait/stop; sin goto). "
    "Unidades→meters: cm/100, m tal cual, pulgadas×0.0254, pies×0.3048. "
    "«gira 45 a la derecha» = grados aunque no digan grados (turn negativo=derecha). "
    "Conectores: primero, después, por último, finalmente, luego… "
    "Zonas nombradas (almacén, válvula…) → unknown. navigate solo: adelante/atras/derecha/izquierda/undock."
)

def to_text(ex):
    user = ex["instruction"]
    assistant = ex["output"]
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant}<|im_end|>\n"
        )
    }

ds = Dataset.from_list(rows).map(to_text)
split = ds.train_test_split(test_size=0.08, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("train", len(train_ds), "eval", len(eval_ds))


In [ ]:
# 3) Modelo + LoRA (Qwen2.5-3B-Instruct, FP16)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# 4) Entrenamiento
from trl import SFTTrainer
from transformers import TrainingArguments

OUT = Path("/content/rbot-operator-lora")
OUT.mkdir(exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUT),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=768,
    tokenizer=tokenizer,
    packing=False,
)
trainer.train()
trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))
print("LoRA guardado en", OUT)


In [ ]:
# 5) Smoke test rápido
import torch

def ask(prompt: str, max_new=180):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip()

for q in [
    "avanza un poco",
    "para atrás",
    "gira a la derecha",
    "detén",
    "gira 45 a la derecha",
    "avanza 12 pulgadas",
    "avanza 2 pies",
    "en primer lugar avanza 0.8 metros después gira 90 a la derecha por último avanza 0.5 metros",
    "primero avanza 80 cm luego gira 90 a la derecha finalmente avanza 50 cm",
    "ve a almacén",
    "ve a la fuga de válvula 3",
    "cuánta batería tiene",
    "estado del robot",
    "cuéntame un chiste",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("---")


In [ ]:
# 6) Merge LoRA → HF completo (local) + copia a Drive
# El error "Transport endpoint is not connected" = Drive FUSE se cayó a mitad
# del save. Solución: guardar en /content y luego copiar (con remount si hace falta).
import gc
import shutil
import time
from peft import PeftModel
from google.colab import drive

# Liberar VRAM del modelo de entrenamiento antes del merge
try:
    del model, trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

LOCAL_MERGE = Path("/content/rbot-operator-merged")
if LOCAL_MERGE.exists():
    shutil.rmtree(LOCAL_MERGE)
LOCAL_MERGE.mkdir(parents=True, exist_ok=True)

# Cargar en GPU sin offload a CPU/meta (evita el warning de safetensors)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, str(OUT))
merged = merged.merge_and_unload()

print("Guardando merge en disco local (rápido/estable)…")
merged.save_pretrained(str(LOCAL_MERGE), safe_serialization=True)
tokenizer.save_pretrained(str(LOCAL_MERGE))
del merged, base
gc.collect()
torch.cuda.empty_cache()
print("Merge local OK:", LOCAL_MERGE)
print("Tamaño:", round(sum(p.stat().st_size for p in LOCAL_MERGE.rglob("*")) / 1e9, 2), "GB")

def remount_drive():
    # remount=True repara mounts FUSE stale (os error 107)
    drive.mount("/content/drive", force_remount=True)
    root = Path("/content/drive/MyDrive/rbot-industrial-ml")
    root.mkdir(parents=True, exist_ok=True)
    return root

DRIVE = remount_drive()
DEST = DRIVE / "rbot-operator-merged"

for attempt in range(1, 4):
    try:
        if DEST.exists():
            shutil.rmtree(DEST)
        print(f"Copiando a Drive (intento {attempt}/3)…")
        shutil.copytree(LOCAL_MERGE, DEST)
        # touch de verificación
        assert any(DEST.glob("*.safetensors")) or any(DEST.glob("*.bin"))
        print("Merge en Drive:", DEST)
        break
    except OSError as e:
        print("Fallo copia Drive:", e)
        if attempt == 3:
            print(
                "⚠️ El merge LOCAL está bien en", LOCAL_MERGE,
                "— descárgalo con files.download de un zip, o re-ejecuta SOLO esta celda."
            )
            raise
        time.sleep(2)
        DRIVE = remount_drive()
        DEST = DRIVE / "rbot-operator-merged"

print("Siguiente: notebook 02_export_gguf_colab.ipynb apuntando a rbot-operator-merged")
